# Calendar Dimension Table

Creates a standard calendar dimension table `{catalog}.retail_gold.dim_calendar` with `date` as the primary key and common calendar attributes. The date range and catalog are controlled via widgets so the job can pass parameters.

In [0]:
# ── Date-range widgets ──
# Defaults cover 2025-01-01 through 2030-12-31: all current transaction
# dates (2026-01 to 2026-05) plus a 4-year future horizon.
# When run as a job task, these widget defaults are overridden by base_parameters.
dbutils.widgets.text("catalog", "retail_q_dab_dev", "Catalog")
dbutils.widgets.text("start_date", "2025-01-01", "Start Date")
dbutils.widgets.text("end_date", "2030-12-31", "End Date")

catalog = dbutils.widgets.get("catalog")
start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")

print(f"Calendar range: {start_date} → {end_date}  (catalog: {catalog})")

In [0]:
# ── Create calendar dimension table ──
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.retail_gold.dim_calendar AS

WITH date_range AS (
    SELECT explode(
        sequence(
            to_date('{start_date}'),
            to_date('{end_date}'),
            interval 1 day
        )
    ) AS date
)

SELECT
    date,
    year(date) AS year,
    quarter(date) AS quarter,
    month(date) AS month,
    date_format(date, 'MMMM') AS month_name,
    date_format(date, 'MMM') AS month_short_name,
    weekofyear(date) AS week_of_year,
    dayofmonth(date) AS day_of_month,
    dayofweek(date) AS day_of_week,
    date_format(date, 'EEEE') AS day_of_week_name,
    date_format(date, 'EEE') AS day_of_week_short_name,
    dayofyear(date) AS day_of_year,

    CASE
        WHEN dayofweek(date) IN (1, 7)
        THEN TRUE
        ELSE FALSE
    END AS is_weekend,

    CASE
        WHEN dayofweek(date) NOT IN (1, 7)
        THEN TRUE
        ELSE FALSE
    END AS is_weekday,

    concat(year(date), '-Q', quarter(date)) AS year_quarter,

    date_format(date, 'yyyy-MM') AS year_month,

    last_day(date) AS last_day_of_month,

    date = last_day(date) AS is_last_day_of_month,

    date = date_trunc('month', date) AS is_first_day_of_month

FROM date_range

ORDER BY date
""")

In [0]:
# ── Validate calendar: no duplicates, correct coverage ──
spark.sql(f"""
SELECT 
    COUNT(*) AS total_dates,
    COUNT(DISTINCT date) AS distinct_dates,
    MIN(date) AS min_date,
    MAX(date) AS max_date
FROM {catalog}.retail_gold.dim_calendar
""").display()